# 데모 1 — LLM API 최소 동작 확인

**목적**: 이 프로젝트의 OpenAI 연동이 실제로 살아 있음을 가장 작은 코드로 증명한다.

두 경로를 차례로 호출한다.

| # | 경로 | 확인하는 것 |
|---|---|---|
| 1 | raw `openai` SDK | API 키 · 네트워크 · 모델이 정상인가 (HTTP 1회) |
| 2 | `agents/shap_agent.py` 의 `SHAPAgent` | 서비스가 실제로 쓰는 LangChain 체인이 도는가 |

모델은 `.env` 의 `LLM_MODEL`(기본 `gpt-4o-mini`)을 따른다.
새 코드를 짜지 않고 **기존 에이전트를 그대로 호출**한다.

In [1]:
"""[0] 환경 준비 — 프로젝트 루트로 이동하고, 이 프로세스에서만 LLM 호출을 허용한다."""
import os, sys, time, json, logging, warnings
from pathlib import Path

# 출력이 읽히도록 잡음만 줄인다(동작에는 영향 없음).
#   - sklearn: DataFrame 대신 ndarray 를 넘길 때 나오는 feature-name 경고가 스텝마다 반복된다
#   - httpx  : OpenAI 요청마다 INFO 로그를 stderr 로 찍는다
warnings.filterwarnings("ignore")
logging.getLogger("httpx").setLevel(logging.WARNING)

# 노트북은 demo/ 에 있지만 프로젝트 코드는 루트 기준 상대경로(models/, data/)를 쓴다.
ROOT = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# .env 의 OPENAI_ENABLED=false 는 24/7 워커의 과금을 막는 의도된 설정이다.
# load_dotenv() 는 이미 설정된 환경변수를 덮어쓰지 않으므로, 먼저 켜 두면
# .env 파일을 건드리지 않고 이 프로세스에서만 LLM 호출을 열 수 있다.
os.environ["OPENAI_ENABLED"] = "true"

from dotenv import load_dotenv
load_dotenv()

# .env 의 키 이름이 OPEN_AI_API_KEY 라서 SDK 가 찾는 OPENAI_API_KEY 로 옮긴다.
# (agents/shap_agent.py 11~12 행과 동일한 처리)
if "OPEN_AI_API_KEY" in os.environ and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = os.environ["OPEN_AI_API_KEY"]

MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")
_key = os.environ.get("OPENAI_API_KEY", "")

print(f"project root   : {ROOT.name}/")
print(f"python         : {sys.version.split()[0]}")
print(f"model          : {MODEL}")
print(f"api key        : {_key[:7]}...{_key[-4:]} (len={len(_key)})")
print(f"OPENAI_ENABLED : {os.getenv('OPENAI_ENABLED')}   <- this process only; .env stays false")


project root   : etch_proj_final/
python         : 3.12.10
model          : gpt-4o-mini
api key        : sk-proj...MFIA (len=164)
OPENAI_ENABLED : true   <- this process only; .env stays false


## 1. raw OpenAI SDK — 가장 작은 호출

LangChain · Neo4j · 모델 파일 전부 빼고, HTTP 요청 한 번으로 API가 응답하는지만 본다.
여기서 실패하면 아래 단계는 볼 필요가 없다.

In [2]:
"""[1] 최소 호출 — openai SDK 직접 사용"""
from openai import OpenAI

client = OpenAI()          # OPENAI_API_KEY 환경변수를 자동으로 읽는다

t0 = time.perf_counter()
resp = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": "너는 반도체 플라즈마 식각 공정 엔지니어다. 두 문장 이내로 답한다."},
        {"role": "user",   "content": "식각 챔버에서 He Chuck 압력이 정상보다 낮아지면 웨이퍼에 어떤 문제가 생기나?"},
    ],
)
lat_raw = time.perf_counter() - t0
usage_raw = resp.usage

print("[answer]")
print(resp.choices[0].message.content)
print()
print(f"model    : {resp.model}")
print(f"latency  : {lat_raw*1000:,.0f} ms")
print(f"tokens   : prompt {usage_raw.prompt_tokens} + completion {usage_raw.completion_tokens}"
      f" = {usage_raw.total_tokens}")

[answer]
He Chuck 압력이 낮아지면 웨이퍼의 고정력이 감소하여 웨이퍼가 식각 공정 중에 이동하거나 진동할 수 있습니다. 이로 인해 식각 품질 저하 및 패턴 손상이 발생할 수 있습니다.

model    : gpt-4o-mini-2024-07-18
latency  : 3,637 ms
tokens   : prompt 66 + completion 56 = 122


## 2. 프로젝트 경로 — `SHAPAgent`

실제 서비스는 `server.py` → `_call_shap_agent()` → `agents/shap_agent.py` 순으로 호출한다.
그 에이전트를 **수정 없이 그대로** 부른다.

입력은 `SHAPExplainer.explain()` 이 내보내는 것과 같은 형식의 고정 샘플이다.
(모델을 실제로 돌려 SHAP 를 계산하는 것은 데모 2에서 한다.)

`SHAPAgent.__init__` 은 `agents/llm_guard.py` 의 `ensure_openai_enabled()` 를 먼저 부른다.
`OPENAI_ENABLED=false` 였다면 **여기서 예외가 나며 과금이 발생하지 않는다.**

In [3]:
"""[2] 프로젝트 경로 호출 — SHAPAgent (LangChain 체인)"""
from agents.shap_agent import SHAPAgent

shap_fixture = [
    {"sensor": "TCP Top Pwr",   "shap_value":  4.7448, "current_value": 390.1,
     "mean_value": 300.0, "normal_range": [285.0, 315.0],
     "status": "High", "direction": "Positive Influence"},
    {"sensor": "TCP Rfl Pwr",   "shap_value":  3.2429, "current_value":  12.4,
     "mean_value":   3.1, "normal_range": [  0.0,   8.0],
     "status": "High", "direction": "Positive Influence"},
    {"sensor": "TCP Impedance", "shap_value": -1.8614, "current_value":  41.2,
     "mean_value":  52.7, "normal_range": [ 48.0,  58.0],
     "status": "Low",  "direction": "Negative Influence"},
]

agent = SHAPAgent()                       # 킬 스위치가 내려가 있으면 여기서 차단된다
t0 = time.perf_counter()
explanation = agent.explain_fault("TCP +30", shap_fixture)
lat_agent = time.perf_counter() - t0

print(f"[SHAPAgent]  {lat_agent:.2f} s\n")
print(explanation)

[SHAPAgent]  9.86 s

결함에 대한 센서 분석 결과를 바탕으로 기술적 분석을 제공합니다. 이번 분석에서는 TCP +30 결함과 관련된 주요 센서 데이터를 검토하였습니다.

1. **TCP Top Pwr (Top Power)**:
   - **현재 값**: 390.1
   - **평균 값**: 300.0
   - **정상 범위**: 285.0 ~ 315.0
   - **상태**: High
   - **SHAP 방향**: Positive Influence

   TCP Top Pwr 센서의 현재 값이 정상 범위를 크게 초과하고 있습니다. 이는 플라즈마 에칭 공정에서 전력 공급이 과도하게 이루어지고 있음을 나타냅니다. 전력이 너무 높으면 플라즈마의 밀도가 증가하여 에칭 속도가 비정상적으로 빨라질 수 있으며, 이는 기판 손상이나 비균일한 에칭을 초래할 수 있습니다. 이 센서의 상태가 'High'로 나타나고 SHAP 값이 긍정적인 영향을 미친 것은 AI가 이 센서의 비정상적인 전력 공급이 결함의 주요 원인이라고 판단했음을 의미합니다.

2. **TCP Rfl Pwr (Reflected Power)**:
   - **현재 값**: 12.4
   - **평균 값**: 3.1
   - **정상 범위**: 0.0 ~ 8.0
   - **상태**: High
   - **SHAP 방향**: Positive Influence

   TCP Rfl Pwr 센서 또한 정상 범위를 초과하고 있으며, 이는 반사 전력이 비정상적으로 높다는 것을 의미합니다. 반사 전력이 높다는 것은 플라즈마가 제대로 형성되지 않거나, 기판과의 상호작용에서 문제가 발생하고 있음을 나타낼 수 있습니다. 이러한 상황은 장비의 안테나 또는 매칭 네트워크에 문제가 있을 수 있음을 시사합니다. 이 센서의 상태가 'High'로 나타나고 SHAP 값이 긍정적인 영향을 미친 것은 이 센서의 비정상적인 반사 전력이 결함의 원인으로 작용하고 있음을 확인해 줍니다.

3. **TCP Impedance (Impeda

## 3. 호출 비용

LangChain 체인은 토큰 사용량을 바로 돌려주지 않으므로 문자 수로 근사한다.
단가는 참고치이며 실제 청구액과 다를 수 있다.

In [4]:
"""[3] 이번 노트북에서 발생한 LLM 비용 요약"""
# gpt-4o-mini 공개 단가 (USD / 1M tokens). 참고치 — 실제 청구액과 다를 수 있다.
PRICE_IN, PRICE_OUT = 0.150, 0.600

est_in  = len(json.dumps(shap_fixture, ensure_ascii=False)) // 4 + 220   # 시스템 프롬프트 포함
est_out = len(explanation) // 2                                          # 한국어는 토큰 밀도가 높다

rows = [
    ("raw openai SDK",   usage_raw.prompt_tokens, usage_raw.completion_tokens, lat_raw,  ""),
    ("SHAPAgent",        est_in,                  est_out,                     lat_agent, "est."),
]

print(f"{'call':<18}{'in':>8}{'out':>8}{'sec':>8}{'USD':>11}   note")
print("-" * 62)
total = 0.0
for name, i, o, lat, note in rows:
    cost = i / 1e6 * PRICE_IN + o / 1e6 * PRICE_OUT
    total += cost
    print(f"{name:<18}{i:>8}{o:>8}{lat:>8.2f}{cost:>11.6f}   {note}")
print("-" * 62)
print(f"{'TOTAL':<18}{'':>8}{'':>8}{'':>8}{total:>11.6f}")
print()
print("OK - LLM API 호출 2건 모두 성공")

call                    in     out     sec        USD   note
--------------------------------------------------------------
raw openai SDK          66      56    3.64   0.000044   
SHAPAgent              352     800    9.86   0.000533   est.
--------------------------------------------------------------
TOTAL                                        0.000576

OK - LLM API 호출 2건 모두 성공
